## 准备数据

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def mnist_dataset(batch_size=128):
    transform = transforms.ToTensor()
    train_ds = datasets.MNIST(root='./mnist/', train=True, transform=transform, download=True)
    test_ds = datasets.MNIST(root='./mnist/', train=False, transform=transform, download=True)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True), DataLoader(test_ds, batch_size=256, shuffle=False)


In [7]:
# CNN in pure PyTorch training loop (basic version)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 建立模型

In [8]:
class myConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(7 * 7 * 64, 10)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = myConvModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## 定义loss以及train loop

In [9]:
def compute_loss(logits, labels):
    return nn.functional.cross_entropy(logits, labels)

def compute_accuracy(logits, labels):
    return (torch.argmax(logits, dim=1) == labels).float().mean()

def train_one_step(model, optimizer, x, y):
    model.train()
    optimizer.zero_grad()
    logits = model(x)
    loss = compute_loss(logits, y)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        acc = compute_accuracy(logits, y)
    return loss, acc

def test(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = compute_loss(logits, y)
        acc = compute_accuracy(logits, y)
    return loss, acc


# 训练

In [10]:
train_ds, test_ds = mnist_dataset()
for epoch in range(2):
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in train_ds:
        x, y = x.to(device), y.to(device)
        loss, acc = train_one_step(model, optimizer, x, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += acc.item() * bs
        n += bs

    test_correct, test_total = 0.0, 0
    with torch.no_grad():
        for x, y in test_ds:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            test_correct += (logits.argmax(dim=1) == y).float().sum().item()
            test_total += y.size(0)

    print('epoch', epoch, ': train loss', total_loss / n, '; train acc', total_acc / n, '; test acc', test_correct / test_total)


epoch 0 : train loss 0.19477482508718968 ; train acc 0.9431833333333334 ; test acc 0.9799
epoch 1 : train loss 0.05365461651484171 ; train acc 0.9837666667302449 ; test acc 0.9884


In [11]:
# expected: test accuracy > 0.96 after a few epochs
